# _lib/rule_engine — Pattern A quarantine routing helpers

Pure helpers for the warn/drop/fail severity model used in Silver.

Drop rules feed two derived helpers:
- `build_clean_predicate(rules)` → SQL string to filter clean vs quarantine.
- `build_failed_rules_expr(rules)` → Spark `Column` producing `ARRAY<STRING>`
  of every drop rule a row violates.

**Import via `%run ./_lib/rule_engine` from the Silver DLT pipeline notebook.**

Reusable for `order_detail` and any future Silver tables that need the same
Pattern A clean/quarantine split.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Column


def build_clean_predicate(drop_rules: dict) -> str:
    """AND-join all drop-rule predicates into a single SQL string.

    A row is clean iff every drop predicate evaluates to TRUE. Empty drop_rules
    returns '1=1' so the caller can apply the result as a filter without
    special-casing.
    """
    if not drop_rules:
        return "1=1"
    return " AND ".join(f"({predicate})" for predicate in drop_rules.values())


def build_failed_rules_expr(drop_rules: dict) -> Column:
    """Return a Column expression producing ARRAY<STRING> of failed rule names.

    For each (name, predicate) pair, emit a CASE WHEN NOT (predicate) THEN 'name'
    column, then collect the names into an array and strip NULLs via
    array_compact (Spark 3.4+ — DBR 13+).

    Returns an empty array for clean rows; in practice only the quarantine
    branch invokes this expression.
    """
    rule_columns = [
        F.when(~F.expr(predicate), F.lit(name))
        for name, predicate in drop_rules.items()
    ]
    return F.array_compact(F.array(*rule_columns))


print("rule_engine loaded: build_clean_predicate, build_failed_rules_expr")